### Data Centers

The plan here to identify a comprehensive list of data centers being built in the US or planning to be built.

We are going to do a dlt -> dbt -> prefect pipeline.

I will try to filter out only relevant forms to keep the data we are storing small.

I will write relevant files probably to a postgres server running on my thinkcentre. Ideally it would be duckdb but I dont think i can write to it over network. 

Motherduck would grow out of the free tier quickly.

The files we keep will be RAG embedded (using a free hugging face model) and stored using PGVECTOR.

From there we can have an agent identify and classify forms that relate to data centers being built, and where they will be built.

In [ ]:
from edgar import set_identity, Filing, get_filings, Company, search_filings, get_current_filings, get_by_accession_number
import pandas as pd
set_identity(user_identity="gwils gwils1414@gmail.com")
#search_filings(query='',forms=["10-K", "10-Q"],start_date='2026-05-01')
#current_10_K = get_current_filings(form="10-K", page_size=None) # All current filings
#current_10_Q = get_current_filings(form="10-Q", page_size=None) # All current filings
#current_8_K = get_current_filings(form="8-K", page_size=None) # All current filings

In [ ]:

def search_data_center_filings():
    '''
    This search will only return 100 results at a time

    We need to run it incrementally lets say by one week increments of start date
    end date

    and store the results somewhere.

    Probably can write them to postgres

    This way we can get 100 each run and end up with the full population
    '''
    data_centers = search_filings("Data Center", 
                                #forms="8-K", 
                                start_date='2025-01-01',
                                end_date='2026-12-31',
                                limit=100)

    identifiers = {'acession_number':[],
                'cik': [],
                'form': [],
                'file_type': []}

    for i in data_centers.results:
        identifiers['acession_number'].append(i.accession_number)
        identifiers['cik'].append(i.cik)
        identifiers['form'].append(i.form)
        identifiers['file_type'].append(i.file_type)

    return identifiers

identifiers = search_data_center_filings()

In [ ]:


def extract_attachment_contents(identifiers:dict):
    file_content = {'name': [],
                    'content': [],
                    'acession_number':[],
                    'form': [],
                    'file_type': [],
                    'attachment_url': []}
    for idx, num in enumerate(identifiers['acession_number']):
        filing = get_by_accession_number(num)
        for attachment in filing.attachments:
            if attachment.document_type == identifiers['file_type'][idx]:
                file_content['filing_url'].append(filing.filing_url)
                file_content['content'].append(filing.text())
                file_content['acession_number'].append(filing.accession_number)
                file_content['form'].append(filing.form)
                file_content['file_type'].append(attachment.document_type)
                file_content['attachment_url'].append(attachment.url)



    return file_content



file_content = extract_attachment_contents(identifiers)  
file_content   



In [ ]:
df = pd.DataFrame(file_content)
df

In [ ]:
document_types = []

for num in acession_numbers[0:20]:
    filing = get_by_accession_number(num)
    for attachment in filing.attachments:
        if attachment.document_type not in document_types:
            document_types.append(attachment.document_type)
        else:
            pass


In [ ]:
for num in acession_numbers[20:50]:
    filing = get_by_accession_number(num)
    for attachment in filing.attachments:
        if attachment.document_type not in document_types:
            document_types.append(attachment.document_type)
        else:
            pass

In [ ]:
for num in acession_numbers[50:80]:
    filing = get_by_accession_number(num)
    for attachment in filing.attachments:
        if attachment.document_type not in document_types:
            document_types.append(attachment.document_type)
        else:
            pass

In [ ]:
for num in acession_numbers[80:100]:
    filing = get_by_accession_number(num)
    for attachment in filing.attachments:
        if attachment.document_type not in document_types:
            document_types.append(attachment.document_type)
        else:
            pass

In [ ]:

counter = 1
for num in acession_numbers:
    filing = get_by_accession_number(num)
    for attachment in filing.attachments:
        if 'EX-99' in attachment.document_type:
            counter += 1

In [ ]:
counter


In [ ]:
for num in acession_numbers[0:50]:
    filing = get_by_accession_number(num)
    for attachment in filing.attachments:
        if 'EX-10' in attachment.document_type:
            print(attachment.text())

EX-10.x — Material Contracts. This is where loan agreements go. Exhibit 10 is the SEC's bucket for material contracts, and credit/loan documents are the classic Exhibit 10 filing: credit agreements, term loan and revolving facilities, promissory notes, security/pledge agreements, guarantees, amendments, and often construction-related contracts (EPC agreements, development agreements, leases for the facility being built). So on your list, essentially the entire run — EX-10.1 through EX-10.43 — is the primary hunting ground. You can't tell from the exhibit number alone which one is the loan; the numbering is just sequential per filer. You have to open them (or read the exhibit index, which gives each a description).

EX-4.x — Debt securities / instruments defining security holder rights. If the "loan" is actually financed through notes or bonds sold to investors (rather than a bank facility), the governing document — an indenture, note, or form of debt security — is filed as Exhibit 4. So EX-4.1 through EX-4.14 are the secondary place to look, specifically when the build is bond-financed. This is the distinction I raised earlier: bank loan → EX-10; debt securities → EX-4 (plus the offering doc itself).

The filings that carry the narrative context (as opposed to the raw contract):

S-1 / S-4 / DRS / DRS-A / 424B3 / 20-F — registration/offering and annual documents. The body of these describes the project, use of proceeds ("we intend to use $X to construct…"), and the debt terms in MD&A and the debt footnotes. The contracts themselves ride along as the EX-10/EX-4 exhibits.
8-K — the event-driven filing for a newly signed material loan or credit facility (Item 1.01 / 2.03), with the agreement attached, again, as an EX-10.
10-K / 10-Q — periodic reports; capex, commitments, and debt discussion in the body.
6-K — foreign private issuers' catch-all; a loan or project announcement could appear here.

In [ ]:
document_types
